In [ ]:
# 1. import thư viện
import torch 
import torch.nn as nn
import torch.optim as optim
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
from google.colab import files 

# 2. Tách train (20 ngày) và test (10 ngày)
uploaded = files.upload()

data = pd.read_csv("data.csv")
print(data.shape)

# Chuyển đổi về dạng cột trong đó x là dữ liệu đầu vào, y là dữ liệu mong muốn ở ngõ ra 
x1 = dataset("N").values
x2 = dataset("N-1").values
x3 = dataset("N-2").values
x4 = dataset("N-3").values
x5 = dataset("N-4").values
y1 = dataset("N+1").values

xx = np.column_stack([x1, x2, x3, x4, x5])
yy = np.column_stack([y1])

# xx: input features, yy: output
n = len(xx) # số lượng mẫu (715)
train_size = int(n * 2/3)
print(train_size)

# Tách train và test
x_train, x_test = xx[:train_size], xx[train_size:]
y_train, y_test = yy[:train_size], yy[train_size:]
print(x_train.shape)

# Đưa sang tensor PyTorch
x_train_tensor = torch.tensor(x_train, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32).view(-1, 1)
x_test_tensor = torch.tensor(x_test, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.float32).view(-1, 1)

# 3. Xây dựng mô hình NN
model = nn.Sequential(
    nn.Linear(5, 64),
    nn.Tanh(),
    nn.Linear(64, 32),
    nn.Tanh(),
    nn.Linear(32, 1)
)

# 4. Hàm loss + optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

# 5. Train 
epochs = 10000
train_losses = []
for epoch in range(epochs):
    y_pred = model(x_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    print(f"Epoch [{epoch}/epochs], Loss: {loss.item():.4f}")

# 6. Dự báo trên test
# model.eval()
# with torch.no_grad():
#   y_test_pred = model(x_test_tensor).squeeze().numpy()

model.eval()
x_input = x_test_tensor[0].clone()
print(x_input)
y_test_pred = []
with torch.no_grad():
    for i in range(N - train_size):
        y_test_onestep = model(x_input).squeeze().numpy()
        x_input = torch.cat([toch.tensor([y_test_onestep], dtype = torch.float32), x_input[:-1]])
        y_test_pred.append(y_test_onestep)
        print(f"step: {i}")
        print(y_test_onestep)
        print(x_input)

# 7. Vẽ kết quả 
plt.figure(figsize = (14, 5))

# Loss 
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title("Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")

# So sánh dự báo
plt.subplot(1, 2, 2)
plt.plot(range(N - train_size), y_test, label = "True Test Data")
plt.plot(range(N- train_size), y_test_pred, label = "Predicted", alpha = 0.8)
plt.title("Temperature Forecasting (Test Data)")
plt.xlabel("Hour (on Test Set)")
plt.ylabel("Temperature (°C)")
plt.legend()

plt.show()